# DeepSeek-Coder-7B Fine-Tuning (Unsloth) → GGUF → Ollama

**Baher | Python Code Analyzer** — Development by Sami Alshammari

This notebook fine-tunes `deepseek-ai/deepseek-coder-7b-instruct-v1.5` on the
CodeAlpaca-20k dataset using Unsloth + LoRA, then exports the result to GGUF
for use with Ollama.

**Run cells top to bottom, in order.** After Cell 1 you MUST restart the
Colab runtime before continuing (Runtime → Restart session), otherwise the
old library versions stay loaded in memory.

## 1. Clean install (fixes version conflicts)

In [ ]:
!pip uninstall -y unsloth unsloth-zoo trl transformers peft accelerate bitsandbytes -q

# Pin transformers to the range Unsloth officially supports (avoid the
# experimental/unstable transformers 5.x path)
!pip install -U "transformers>4.51.3,<=4.57.6" -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install -U trl peft accelerate bitsandbytes -q

print("Done. Now go to Runtime -> Restart session, then run the next cell.")

## 2. STOP — Restart the runtime now

`Runtime` → `Restart session`, then continue from the cell below.

In [ ]:
import trl, transformers, unsloth
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)
print("unsloth:", unsloth.__version__)

## 3. Load base model and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch
from trl import SFTConfig, SFTTrainer

context_window = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "deepseek-ai/deepseek-coder-7b-instruct-v1.5",
    max_seq_length = context_window,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Load and format the dataset

In [ ]:
dataset = load_dataset("sahil2801/CodeAlpaca-20k", split = "train")
print(dataset[0])

prompt_format = """You are an expert AI code assistant. Your task is to analyze, explain, or optimize the given code.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = prompt_format.format(instruction, input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

## 5. Train (SFT with LoRA)

In [ ]:
sft_config = SFTConfig(
    dataset_text_field = "text",
    max_length = context_window,
    dataset_num_proc = 2,
    packing = False,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    save_strategy = "no",
)

trainer = SFTTrainer(
    model = model,
    args = sft_config,
    train_dataset = dataset,
)

trainer_stats = trainer.train()

## 6. Save the LoRA adapter immediately 

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Saved locally to /content/lora_model")

## 7. Check free disk space before GGUF export

In [ ]:
!df -h /content


## 8. Export to GGUF (q4_k_m quantization)

In [ ]:
# Uses the model already in memory from training above — no need to
# re-load from a Hub repo name that doesn't exist.
model.save_pretrained_gguf(
    "my_deepseek_model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

## 9. (Only if running GGUF export in a NEW session)

If you closed the training session and are exporting later, reload from your saved `lora_model` folder — never from a name that was never saved.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",   # your local saved folder, not "my-deepseek"
    max_seq_length = 2048,
    load_in_4bit = True,
)

model.save_pretrained_gguf("my_deepseek_model_gguf", tokenizer, quantization_method = "q4_k_m")

## 10. Download the GGUF file to your local machine

In [ ]:
from google.colab import files
import glob

gguf_path = glob.glob("my_deepseek_model_gguf/*.gguf")[0]
print("Found:", gguf_path)
files.download(gguf_path)

## 11. Load into Ollama (run these on your own machine, not in Colab)

1. Put the downloaded `.gguf` file in a folder, e.g. `~/models/my-deepseek/`
2. In that folder, create a file named `Modelfile` (no extension) with this content:

```
FROM ./deepseek-coder-7b-instruct-v1.5.Q4_K_M.gguf

TEMPLATE """You are an expert AI code assistant. Your task is to analyze, explain, or optimize the given code.

### Instruction:
{{ .Prompt }}

### Response:
"""

PARAMETER stop "### Instruction:"
PARAMETER temperature 0.2
```

3. Then run:

```bash
cd ~/models/my-deepseek/
ollama create my-deepseek -f Modelfile
ollama run my-deepseek
```

4. Verify it shows up:

```bash
ollama list
```

The name must match exactly `my-deepseek` — that's what both the HTML
frontend and the FastAPI backend below expect.

## 12. Local FastAPI Gateway Microservice

Save the python service below as a standalone file named `app.py` on the machine where Ollama is running. Then start the server using:

```bash
pip install fastapi uvicorn ollama
uvicorn app:app --host 0.0.0.0 --port 8000
```

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import ollama

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    message: str

@app.post("/chat")
async def chat_endpoint(request: ChatRequest):
    try:
        response = ollama.chat(model='my-deepseek', messages=[
            {
                'role': 'user',
                'content': request.message,
            },
        ])
        return {"response": response['message']['content']}
    except Exception as e:
        return {"response": f"error: {str(e)}"}
```

A ready-to-run copy of this is also provided as `app.py` alongside this
notebook.